In [ ]:
import itertools

import stim
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.qubit_array import QubitArray
from utils.simulation.clifft import sample

In [ ]:
def make_base_circuitry(preparation: Pauli):
    qubits = QubitArray(dimensions=(1, 2))
    circuitry = Circuitry(qubits, clifford=False)

    circuitry.append(f"R{preparation.name}", 0)
    circuitry.append("RX", 1)
    circuitry.append("S", 1)
    circuitry.append("CX", [1, 0])

    return qubits, circuitry


def format(preparation: Pauli, observable: Pauli):
    return rf"$\overline{{\mathbf{{{preparation.name}}}}}/\overline{{\mathbf{{{observable.name}}}}}$"

In [ ]:
display_url = True

In [ ]:
scenarios_direct = {}

for preparation, observable in itertools.product(Pauli, repeat=2):
    qubits = QubitArray(dimensions=(1, 1))
    circuitry = Circuitry(qubits, clifford=False)

    circuitry.append(f"R{preparation.name}", 0)
    circuitry.append("S", 0)

    circuitry.append_observable(0, "OBSERVABLE", {0: observable.name})

    scenario = rf"$\overline{{\mathbf{{{preparation.name}}}}}/\overline{{\mathbf{{{observable.name}}}}}$"
    scenarios_direct[scenario] = circuitry

    if display_url:
        display(
            Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
        )

In [ ]:
scenarios_correct = {}

for preparation, observable in itertools.product(Pauli, repeat=2):
    _, circuitry = make_base_circuitry(preparation)

    # Correct both error types (X, Z).
    circuitry.append("CX", [0, 1])
    circuitry.append("CZ", [0, 1])

    circuitry.append_observable(0, "OBSERVABLE", {1: observable.name})

    scenario = format(preparation, observable)
    scenarios_correct[scenario] = circuitry

    if display_url:
        display(
            Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
        )

In [ ]:
scenarios_feedforward = {}

for preparation, observable in itertools.product(Pauli, repeat=2):
    qubits, circuitry = make_base_circuitry(preparation)
    circuitry.append("MZ", 0)
    qubits.record_measurement(0, "MZ0")

    # Correct X-type error using feed-forward.
    circuitry.append("CX", [stim.target_rec(-1), 1])
    # Correct Z-type error using feed-forward.
    circuitry.append("CZ", [stim.target_rec(-1), 1])

    circuitry.append_observable(0, "OBSERVABLE", {1: observable.name})

    scenario = format(preparation, observable)
    scenarios_feedforward[scenario] = circuitry

    if display_url:
        display(
            Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
        )

In [ ]:
scenarios_tracked = {}

for preparation, observable in itertools.product(Pauli, repeat=2):
    qubits, circuitry = make_base_circuitry(preparation)
    circuitry.append("MZ", 0)
    qubits.record_measurement(0, "MZ0")

    # Correct Z-type error.
    circuitry.append("CZ", [stim.target_rec(-1), 1])

    # Manually track the X-error.
    if observable == Pauli.X:
        # The X-observable is not affected by an X-error
        circuitry.append_observable(0, "OBSERVABLE", {1: observable.name})
    else:
        # The Y-observable and Z-observable are affected by an X-error
        circuitry.append_observable(0, "OBSERVABLE", {1: observable.name}, "MZ0")

    scenario = format(preparation, observable)
    scenarios_tracked[scenario] = circuitry

    if display_url:
        display(
            Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
        )

In [ ]:
title = r"S-gate direct"
sample(
    scenarios_direct,
    correction=True,
    title=title,
    label="Preparation/Observable",
    fontsize=8,
    figsize=(16, 5),
)

In [ ]:
title = r"S-gate construct [correct]"
sample(
    scenarios_correct,
    correction=True,
    title=title,
    label="Preparation/Observable",
    fontsize=8,
    figsize=(16, 5),
)

In [ ]:
title = r"S-gate construct [feed-forward]"
sample(
    scenarios_feedforward,
    correction=True,
    title=title,
    label="Preparation/Observable",
    fontsize=8,
    figsize=(16, 5),
)

In [ ]:
title = r"S-gate construct [tracked]"
sample(
    scenarios_tracked,
    correction=True,
    title=title,
    label="Preparation/Observable",
    fontsize=8,
    figsize=(16, 5),
)